In [2]:
# ─── CELL 1: mount & navigate ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Multi-Agent-RAG-System-for-Enterprise-Document-Intelligence")
print("Working dir:", os.getcwd())

Mounted at /content/drive
Working dir: /content/drive/MyDrive/Multi-Agent-RAG-System-for-Enterprise-Document-Intelligence


In [3]:
# ─── CELL 2: install ────────────────────────────────────────────────────
!pip install langchain langchain-community langchain-core \
             langchain-groq langgraph groq \
             faiss-cpu sentence-transformers \
             datasets pandas numpy matplotlib \
             fastapi uvicorn python-dotenv -q
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Done.


In [4]:
# ─── CELL 3: create config ───────────────────────────────────────────────
config_content = '''# src/config.py
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
MODEL_NAME   = "llama-3.1-8b-instant"
TEMPERATURE  = 0.1
MAX_TOKENS   = 512
'''

with open("src/config.py", "w") as f:
    f.write(config_content)
print("config.py created.")

config.py created.


In [5]:
# ─── CELL 4: load MS MARCO dataset ──────────────────────────────────────
from datasets import load_dataset
import pandas as pd
import os

os.makedirs("data/processed", exist_ok=True)

print("Loading MS MARCO...")
ds = load_dataset("microsoft/ms_marco", "v1.1", split="train[:500]")

docs     = []
qa_pairs = []

for i, sample in enumerate(ds):
    query    = sample['query']
    passages = sample['passages']

    for j, (text, is_selected) in enumerate(
        zip(passages['passage_text'], passages['is_selected'])
    ):
        if text.strip():
            docs.append({
                'doc_id':   f"{i}_{j}",
                'text':     text.strip(),
                'query':    query,
                'selected': is_selected
            })

    answers = sample.get('answers', [])
    if answers and answers[0] != 'No Answer Present.':
        qa_pairs.append({
            'id':       i,
            'question': query,
            'answer':   answers[0]
        })

df_docs = pd.DataFrame(docs)
df_qa   = pd.DataFrame(qa_pairs)

df_docs.to_csv("data/processed/documents.csv", index=False)
df_qa.to_csv("data/processed/qa_pairs.csv",    index=False)

print(f"Documents: {len(df_docs)}")
print(f"QA pairs:  {len(df_qa)}")
print(f"\nSample document:\n{df_docs.iloc[0]['text'][:300]}")
print(f"\nSample QA:\n{df_qa.iloc[0]}")

Loading MS MARCO...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/9.48k [00:00<?, ?B/s]

v1.1/validation-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

v1.1/train-00000-of-00001.parquet:   0%|          | 0.00/175M [00:00<?, ?B/s]

v1.1/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10047 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/82326 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9650 [00:00<?, ? examples/s]

Documents: 4089
QA pairs:  491

Sample document:
Since 2007, the RBA's outstanding reputation has been affected by the 'Securency' or NPA scandal. These RBA subsidiaries were involved in bribing overseas officials so that Australia might win lucrative note-printing contracts. The assets of the bank include the gold and foreign exchange reserves of

Sample QA:
id                                                          0
question                                          what is rba
answer      Results-Based Accountability is a disciplined ...
Name: 0, dtype: object


In [6]:
# ─── CELL 5: build and cache embeddings ──────────────────────────────────
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

EMBEDDINGS_PATH = "data/processed/embeddings.npy"

embedder = SentenceTransformer("all-MiniLM-L6-v2")

if os.path.exists(EMBEDDINGS_PATH):
    print("Loading cached embeddings...")
    embeddings = np.load(EMBEDDINGS_PATH)
else:
    print("Computing embeddings...")
    texts      = df_docs['text'].tolist()
    embeddings = embedder.encode(
        texts, show_progress_bar=True, batch_size=64)
    embeddings = np.array(embeddings).astype('float32')
    np.save(EMBEDDINGS_PATH, embeddings)
    print("Embeddings cached.")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings.astype('float32'))
print(f"FAISS index: {index.ntotal} vectors, dim={embeddings.shape[1]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading cached embeddings...
FAISS index: 4089 vectors, dim=384


In [7]:
# ─── CELL 6: create retrieval tool ───────────────────────────────────────
retrieval_tool_content = '''# tools/retrieval_tool.py
"""
Vector retrieval tool — MCP-style tool interface.
Used by the Retriever Agent to search the document corpus.
"""
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from langchain.tools import tool

EMBEDDINGS_PATH = "data/processed/embeddings.npy"

df_docs  = pd.read_csv("data/processed/documents.csv")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

if os.path.exists(EMBEDDINGS_PATH):
    embeddings = np.load(EMBEDDINGS_PATH)
else:
    texts      = df_docs["text"].tolist()
    embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=64)
    embeddings = np.array(embeddings).astype("float32")
    np.save(EMBEDDINGS_PATH, embeddings)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings.astype("float32"))
print(f"Retrieval tool: index built with {index.ntotal} vectors")


@tool
def search_documents(query: str) -> str:
    """
    Search the document knowledge base for passages relevant to the query.
    Use this tool to find factual information from the document corpus.
    Input: natural language search query
    Output: top 3 relevant document passages
    """
    q_emb              = embedder.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, 3)

    results = []
    for rank, idx in enumerate(indices[0]):
        text = df_docs.iloc[idx]["text"]
        results.append(f"[Document {rank+1}]\\n{text[:500]}")

    return "\\n\\n".join(results)
'''

with open("tools/retrieval_tool.py", "w") as f:
    f.write(retrieval_tool_content)
print("retrieval_tool.py created.")

retrieval_tool.py created.


In [8]:
# ─── CELL 7: create calculator and summarise tools ───────────────────────
calc_content = '''# tools/calculator_tool.py
import math
from langchain.tools import tool

@tool
def calculate(expression: str) -> str:
    """
    Evaluate a mathematical expression.
    Input: plain math expression e.g. 45 * 14
    Output: numerical result
    """
    try:
        expression = expression.strip().strip("\\'\\\"")
        allowed = {k: getattr(math, k) for k in dir(math)
                   if not k.startswith("_")}
        allowed["__builtins__"] = {}
        result = eval(expression, allowed)
        return f"Result: {result}"
    except Exception as e:
        return f"Calculation error: {e}"
'''

summarise_content = '''# tools/summarise_tool.py
from langchain.tools import tool

@tool
def summarise_text(text: str) -> str:
    """
    Summarise a long text passage into 2-3 concise sentences.
    Input: full text passage to summarise
    Output: 2-3 sentence summary
    """
    if text.strip().startswith("[Document") and len(text.strip()) < 20:
        return "Error: pass full text, not a label like [Document 1]"
    sentences = text.replace("\\n", " ").split(".")
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    summary   = ". ".join(sentences[:3])
    return summary + "." if summary else text[:300]
'''

with open("tools/calculator_tool.py", "w") as f:
    f.write(calc_content)
with open("tools/summarise_tool.py", "w") as f:
    f.write(summarise_content)
print("Tools created.")

Tools created.


In [9]:
# ─── CELL 8: test retrieval tool ─────────────────────────────────────────
import sys
sys.path.insert(0, '.')

from tools.retrieval_tool import search_documents

result = search_documents.invoke("what is the Reserve Bank of Australia")
print(result[:400])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieval tool: index built with 4089 vectors
[Document 1]
The Reserve Bank of Australia (RBA) came into being on 14 January 1960 as Australia 's central bank and banknote issuing authority, when the Reserve Bank Act 1959 removed the central banking functions from the Commonwealth Bank. The assets of the bank include the gold and foreign exchange reserves of Australia, which is estimated to have a net worth of A$101 billion. Nearly 94% of the


In [10]:
# ─── CELL 9: commit────────────────────────────────────────────────
!git config --global user.email "chaitanyamhetre97@gmail.com"
!git config --global user.name "chaitanyamhetre"
!git add .
!git commit -m "Data loaded, embeddings cached, tools created"
!git push
print("Pushed.")

[main b452bc4] Data loaded, embeddings cached, tools created
 5 files changed, 93 insertions(+)
 create mode 100644 notebooks/01_data_and_setup.ipynb
 create mode 100644 src/config.py
 create mode 100644 tools/calculator_tool.py
 create mode 100644 tools/retrieval_tool.py
 create mode 100644 tools/summarise_tool.py
Enumerating objects: 14, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (10/10), done.
Writing objects: 100% (10/10), 19.35 KiB | 683.00 KiB/s, done.
Total 10 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/chaitanyamhetre/Multi-Agent-RAG-System-for-Enterprise-Document-Intelligence.git
   ca5d35b..b452bc4  main -> main
Pushed.
